# CozyDay P1 A8 — Stage 1
**Owner: Lin Gao** | Step 1.1 Knowledge Base and Step 1.2 Chunking only. Stages 2–4 will be added by teammates after Stage 1 is merged.

## 1. Load and validate the knowledge base
30 original English paragraphs with stable IDs, titles, topics, and bodies. All filesystem paths are relative to this notebook / module directory.

In [1]:
from chunking import (load_knowledge_base, validate_knowledge_base,
                      fixed_length_chunks, overlapping_paragraph_chunks, hybrid_topic_chunks,
                      build_all_chunks, validate_chunks)
records = load_knowledge_base()
validation = validate_knowledge_base(records)
print(validation)
print("First paragraph:", records[0]["id"], records[0]["title"])

{'paragraphs': 30, 'total_words': 3111, 'min_words': 97, 'max_words': 112, 'unique_ids': 30}
First paragraph: P001 Creating an actionable task


## 2. Three deterministic chunking strategies
**Fixed:** whole-sentence packing to 130 words, flushing at fine-topic boundaries. **Overlap:** adjacent whole-paragraph pairs inside documented related sections, stride 1. **Hybrid:** nonoverlapping whole paragraphs with identical fine topics, soft cap 230 words.

All examples below show complete chunk text. See `rag_analysis.md` for the semantic section mapping and soft-cap exceptions.

In [2]:
chunks_by_strategy = build_all_chunks(records)
for strategy, chunks in chunks_by_strategy.items():
    print(f"{strategy}: {len(chunks)} chunks")
    example = chunks[0]
    print("ID:", example["chunk_id"], "| Topic:", example["topic"])
    print("Source IDs:", example["source_ids"], "| Words:", example["word_count"])
    print("Complete example:\n" + example["text"] + "\n")
print("Actual overlapping windows in Task Organization:")
for chunk in chunks_by_strategy["overlap"][:3]:
    print(chunk["chunk_id"], chunk["source_ids"], chunk["topic"])

fixed: 30 chunks
ID: FIXED_001 | Topic: Task Creation
Source IDs: ['P001'] | Words: 112
Complete example:
A useful task describes one clear outcome rather than a vague intention. For example, replace “work on biology” with “summarize chapter four and answer the practice questions.” When preparing a task in CozyDay, give it a short descriptive title, optional notes, an estimated duration, and a realistic due date if one exists. Record the next physical action in the notes when the work feels overwhelming. Large assignments should become several smaller tasks, each with an observable completion criterion. Review a newly entered task to make sure its wording still makes sense when seen without the surrounding conversation. An organized task list supports planning, but creating tasks does not replace doing the work.

overlap: 19 chunks
ID: OVERLAP_001 | Topic: Task Creation
Source IDs: ['P001', 'P002'] | Words: 216
Complete example:
A useful task describes one clear outcome rather than a v

## 3. Automated checks and Stage 2 interface
Validation checks text preservation, exact source provenance, metadata, semantic boundaries, and coverage. The standalone regression suite is run with `python test_stage1.py`. Stage 2 embeds `chunk["text"]` and preserves the other fields. Full-corpus size validation is separate from chunking so a small scaling subset remains supported.

In [3]:
for strategy, chunks in chunks_by_strategy.items():
    validate_chunks(records, chunks, strategy)
print("Stage 1 checks PASSED")
print("Chunk object keys:", list(chunks_by_strategy["fixed"][0]))
print("Five-paragraph scaling subset:",
      {name: len(chunks) for name, chunks in build_all_chunks(records[:5]).items()})

Stage 1 checks PASSED
Chunk object keys: ['chunk_id', 'strategy', 'source_ids', 'topic', 'text', 'word_count']
Five-paragraph scaling subset: {'fixed': 5, 'overlap': 4, 'hybrid': 3}
